# Autoresearch Experiment Analysis

Analysis of the autonomous Scrabble search experiments logged in `results.tsv`. The metric is `best_score`: the highest-scoring single play found in a 5-minute run (higher is better).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the TSV (tab-separated, 5 columns: commit, best_score, best_word, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["best_score"] = pd.to_numeric(df["best_score"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  score={row['best_score']:5.0f}  {row['best_word']:<15s}  {row['description']}")

## Best Score Over Time

Track how the best (kept) score evolves as experiments progress. The running maximum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["best_score"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_score = valid.loc[kept_mask, "best_score"]
ax.scatter(kept_idx, kept_score,
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line
running_max = kept_score.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its word and description
for idx, score in zip(kept_idx, kept_score):
    desc = f"{valid.loc[idx, 'best_word']}: {str(valid.loc[idx, 'description']).strip()}"
    if len(desc) > 55:
        desc = desc[:52] + "..."

    ax.annotate(desc, (idx, score),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Best single-play score (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_score = df.iloc[0]["best_score"]
best_row = kept.loc[kept["best_score"].idxmax()]
best_score = best_row["best_score"]

print(f"Baseline best_score: {baseline_score:.0f}")
print(f"Best best_score:     {best_score:.0f} ({best_row['best_word']})")
print(f"Total improvement:   {best_score - baseline_score:+.0f} ({(best_score - baseline_score) / baseline_score * 100:.1f}%)")
print(f"Best experiment:     {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: score={row['best_score']:5.0f}  {row['best_word']:<15s}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's score
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_score"] = kept["best_score"].shift(1)
kept["delta"] = kept["best_score"] - kept["prev_score"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>6}  {'Score':>6}  {'Word':<15s}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+6.0f}  {row['best_score']:6.0f}  {row['best_word']:<15s}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+6.0f}  {'':>6}  {'':<15s}  TOTAL improvement over baseline")